# Forward and backward stepwise modeling



## Setup


In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold
from palmerpenguins import load_penguins
from plotnine import *
import statsmodels.api as sm

## Data


In [2]:
penguins = load_penguins().dropna()
penguins.columns

Index(['species', 'island', 'bill_length_mm', 'bill_depth_mm',
       'flipper_length_mm', 'body_mass_g', 'sex', 'year'],
      dtype='str')

## Regression

### Data Prep

In [3]:
mass_features = ["bill_length_mm", "bill_depth_mm", "species", 
                 "flipper_length_mm", "island", "sex"]
X_mass = (
    pd.get_dummies(penguins[mass_features], dtype = int, drop_first=True)
) 
y_mass = penguins["body_mass_g"]

### Cross-Validation Setup

In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

### Forward Selection Set Up

In [5]:
selector = SequentialFeatureSelector(
    estimator=LinearRegression(),
    direction="forward",               
    n_features_to_select=4,
    scoring="neg_mean_squared_error",
    cv=cv,
)

### Selecting Predictors

In [6]:
selector.fit(X_mass, y_mass)

,estimator estimator: estimator instanceAn unfitted estimator.,LinearRegression()
,"n_features_to_select n_features_to_select: ""auto"", int or float, default=""auto""If `""auto""`, the behaviour depends on the `tol` parameter:- if `tol` is not `None`, then features are selected while the score change does not exceed `tol`.- otherwise, half of the features are selected.If integer, the parameter is the absolute number of features to select.If float between 0 and 1, it is the fraction of features to select... versionadded:: 1.1 The option `""auto""` was added in version 1.1... versionchanged:: 1.3 The default changed from `""warn""` to `""auto""` in 1.3.",4
,"scoring scoring: str or callable, default=NoneScoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)`` that returns a single value. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. In all othercases, :class:`~sklearn.model_selection.KFold` is used. These splittersare instantiated with `shuffle=False` so the splits will be the sameacross calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.",KFold(n_split... shuffle=True)
,"tol tol: float, default=NoneIf the score is not incremented by at least `tol` between twoconsecutive feature additions or removals, stop adding or removing.`tol` can be negative when removing features using `direction=""backward""`.`tol` is required to be strictly positive when doing forward selection.It can be useful to reduce the number of features at the cost of a smalldecrease in the score.`tol` is enabled only when `n_features_to_select` is `""auto""`... versionadded:: 1.1",None
,"direction direction: {'forward', 'backward'}, default='forward'Whether to perform forward selection or backward selection.",'forward'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel. When evaluating a new feature toadd or remove, the cross-validation procedure is parallel over thefolds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['bill_length_mm','bill_depth_mm','flipper_length_mm',...,'island_Dream', 'island_Torgersen','sex_male']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,8
n_features_to_select_ n_features_to_select_: intThe number of features that were selected.,int64,np.int64(4)


### Selected Predictors 

In [7]:
selected_features = X_mass.columns[selector.get_support()]
print(selected_features.tolist())

['bill_depth_mm', 'flipper_length_mm', 'species_Gentoo', 'sex_male']


### Fit Model

In [8]:
xx = sm.add_constant(X_mass[selected_features])
model = sm.OLS(y_mass, xx).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            body_mass_g   R-squared:                       0.871
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     555.0
Date:                Mon, 19 Oct 2026   Prob (F-statistic):          1.39e-144
Time:                        14:27:44   Log-Likelihood:                -2358.8
No. Observations:                 333   AIC:                             4728.
Df Residuals:                     328   BIC:                             4747.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              -936.3793    546.757     -1.713      0.088   -2011.972     139.213
bill_depth_mm        76.1216     19.743      3.856      0.000      37.283     114.960
flipper_length_mm    15.7892      2.690      5.870      0.000      10.498      21.081
species_Gentoo     1229.2380    110.694     11.105      0.000    1011.478    1446.998
sex_male            444.8401     44.606      9.973      0.000     357.091     532.590
==============================================================================
Omnibus:                        0.429   Durbin-Watson:                   2.133
Prob(Omnibus):                  0.807   Jarque-Bera (JB):                0.437
Skew:                           0.086   Prob(JB):                        0.804
Kurtosis:                       2.958   Cond. No.                     6.96e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 6.96e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Problem

Fit a backward stepwise Regression model up 5 features. Use a 10-fold cross-validation approach.

## Classification

### Data Prep

In [20]:
species_features = ["bill_length_mm", "bill_depth_mm", "body_mass_g", 
                 "flipper_length_mm"]
X_species = penguins[species_features]
y_species = pd.Categorical(penguins["species"])

### Cross Validation Set Up

In [24]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

### Backward Selection Set Up

In [25]:
selector = SequentialFeatureSelector(
    estimator=LogisticRegression(max_iter=10000),
    direction="backward",               
    n_features_to_select=2,
    scoring="accuracy",
    cv=cv,
)

### Selecting Predictors

In [26]:
selector.fit(X_species, y_species)

,estimator estimator: estimator instanceAn unfitted estimator.,LogisticRegre...ax_iter=10000)
,"n_features_to_select n_features_to_select: ""auto"", int or float, default=""auto""If `""auto""`, the behaviour depends on the `tol` parameter:- if `tol` is not `None`, then features are selected while the score change does not exceed `tol`.- otherwise, half of the features are selected.If integer, the parameter is the absolute number of features to select.If float between 0 and 1, it is the fraction of features to select... versionadded:: 1.1 The option `""auto""` was added in version 1.1... versionchanged:: 1.3 The default changed from `""warn""` to `""auto""` in 1.3.",2
,"direction direction: {'forward', 'backward'}, default='forward'Whether to perform forward selection or backward selection.",'backward'
,"scoring scoring: str or callable, default=NoneScoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)`` that returns a single value. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. In all othercases, :class:`~sklearn.model_selection.KFold` is used. These splittersare instantiated with `shuffle=False` so the splits will be the sameacross calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.",KFold(n_split... shuffle=True)
,"tol tol: float, default=NoneIf the score is not incremented by at least `tol` between twoconsecutive feature additions or removals, stop adding or removing.`tol` can be negative when removing features using `direction=""backward""`.`tol` is required to be strictly positive when doing forward selection.It can be useful to reduce the number of features at the cost of a smalldecrease in the score.`tol` is enabled only when `n_features_to_select` is `""auto""`... versionadded:: 1.1",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel. When evaluating a new feature toadd or remove, the cross-validation procedure is parallel over thefolds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](4,)","['bill_length_mm','bill_depth_mm','body_mass_g','flipper_length_mm']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,4
n_features_to_select_ n_features_to_select_: intThe number of features that were selected.,int64,np.int64(2)


### Selected Predictors

In [22]:
selected_features = X_species.columns[selector.get_support()]
print(selected_features.tolist())

['bill_length_mm', 'bill_depth_mm']


### Fitted Model

In [23]:
xx = sm.add_constant(X_species[selected_features])
model = sm.MNLogit(y_species, xx).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.071740
         Iterations 12


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                      y   No. Observations:                  333
Model:                        MNLogit   Df Residuals:                      327
Method:                           MLE   Df Model:                            4
Date:                Mon, 19 Oct 2026   Pseudo R-squ.:                  0.9319
Time:                        14:31:58   Log-Likelihood:                -23.889
converged:                       True   LL-Null:                       -350.86
Covariance Type:            nonrobust   LLR p-value:                3.259e-140
==================================================================================
   y=Chinstrap       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            -24.3847     13.527     -1.803      0.071     -50.897       2.127
bill_length_mm     2.2063      0.686      3.218      0.001       0.863       3.550
bill_depth_mm     -3.9759      1.486     -2.676      0.007      -6.888      -1.063
----------------------------------------------------------------------------------
      y=Gentoo       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             25.0153     20.134      1.242      0.214     -14.447      64.478
bill_length_mm     2.6928      0.704      3.827      0.000       1.314       4.072
bill_depth_mm     -8.3213      1.818     -4.578      0.000     -11.884      -4.759
==================================================================================
"""